# Model 3 · Deepnet over MPNet Pair Features

Supervised MLP head over `[q ; o ; q*o ; |q-o|]` pair features built from the
sentence-transformer embeddings. Includes Model 2's cells as a prerequisite
(needed for `Q_train`/`O_train`/`EMB_DIM`).


In [ ]:
import os, re, gc, math, random, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config - CPU only
# ---------------------------------------------------------------------------
BASE        = "../data"
TRAIN_PATH  = f"{BASE}/train.csv"
TEST_PATH   = f"{BASE}/test.csv"
OUTPUT_PATH = "../outputs/submission.csv"

OPTIONS  = ["A", "B", "C", "D", "E"]
SEED     = 42
VAL_SIZE = 0.20
N_FOLDS  = 5

MPNET_ID = "sentence-transformers/all-mpnet-base-v2"

# ---------------------------------------------------------------------------
# Weights & Biases - one run per model, so every model has a tracked run with
# comparable metrics (MAP@3, accuracy, macro F1, weighted F1).
# Wrapped so that a W&B failure can never abort the run or lose the submission.
# ---------------------------------------------------------------------------
WANDB_PROJECT = "23f2004742-t22026"
WANDB_ON = True

os.environ["WANDB_SILENT"] = "true"

try:
    import wandb
    _key = None
    try:
        from kaggle_secrets import UserSecretsClient
        _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        _key = os.environ.get("WANDB_API_KEY")

    if _key:
        wandb.login(key=_key)
        print(f"W&B ready -> project '{WANDB_PROJECT}'")
    else:
        # A bare wandb.login() waits on stdin, which would hang a
        # "Save & Run All" commit forever. Disable instead of risking that.
        WANDB_ON = False
        print("W&B key not found (add WANDB_API_KEY as a Kaggle secret).")
        print("Continuing without tracking; all metrics are still printed.")
except Exception as e:
    WANDB_ON = False
    print(f"W&B unavailable ({type(e).__name__}); metrics still printed locally")

# DeBERTa is kept as an experiment only. It is a 435M model: fine-tuning it on
# CPU is not practical, so it stays off unless a GPU is attached.
RUN_DEBERTA = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"DeBERTa experiment: {'ON' if RUN_DEBERTA else 'OFF (no GPU - expected on CPU)'}")


## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

counts = train_df["answer"].value_counts().reindex(OPTIONS)
probs  = counts / counts.sum()
order  = list(probs.sort_values(ascending=False).index)

PRIOR_MAP3 = probs[order[0]] + probs[order[1]] / 2 + probs[order[2]] / 3
LABEL_LOGPRIOR = np.log(probs[OPTIONS].values.astype(np.float64))

print("\nAnswer distribution:")
for o in order:
    print(f"  {o}  {counts[o]:>4}  ({probs[o]:.1%})")
print(f"\nRandom-ordering MAP@3        : 0.3667")
print(f"Always '{' '.join(order[:3])}' MAP@3          : {PRIOR_MAP3:.4f}   <- the bar to beat")

train_df.head(3)


## 2. Preprocessing and split

Prompts carry boilerplate prefixes ("Pick the best possible answer:", …) which
are stripped. **The split is made on the CLEANED prompt**, because two prompts
that differ only by prefix become identical after cleaning - splitting on raw
text would put the same question on both sides and make validation meaningless.


In [ ]:
BOILERPLATE = [
    r"^Pick the best possible answer:\s*", r"^Choose the correct answer:\s*",
    r"^Select the most accurate option:\s*", r"^Identify the correct statement:\s*",
    r"^Determine the correct option:\s*", r"^Which of the following\s*",
    r"\s*among the listed options\.?$", r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$", r"\s*carefully\.?$",
]


def clean_text(t):
    t = re.sub(r"\s+", " ", str(t)).strip()
    for p in BOILERPLATE:
        t = re.sub(p, "", t, flags=re.IGNORECASE).strip()
    return t


def clean_frame(df):
    out = df.copy()
    for c in ["prompt"] + OPTIONS:
        out[c] = out[c].map(clean_text)
    return out


train = clean_frame(train_df)
test  = clean_frame(test_df)

n_raw, n_clean = train_df["prompt"].nunique(), train["prompt"].nunique()
print(f"Unique prompts  raw {n_raw}  ->  cleaned {n_clean}   "
      f"({n_raw - n_clean} collapsed by preprocessing)")

uniq = train["prompt"].unique()
tr_p, va_p = train_test_split(uniq, test_size=VAL_SIZE, random_state=SEED)

train_split = train[train["prompt"].isin(tr_p)].reset_index(drop=True)
val_split   = train[train["prompt"].isin(va_p)].reset_index(drop=True)
assert not (set(train_split["prompt"]) & set(val_split["prompt"]))

y_val   = val_split["answer"].tolist()
y_train = train["answer"].tolist()

print(f"Train {len(train_split)} | Val {len(val_split)} | no cleaned-prompt overlap")


## 3. Metrics - MAP@3, Accuracy, Macro F1

In [ ]:
def average_precision_at_3(actual, predicted):
    for rank, p in enumerate(predicted[:3], start=1):
        if p == actual:
            return 1.0 / rank
    return 0.0


def map_at_3(actuals, preds):
    return float(np.mean([average_precision_at_3(a, p) for a, p in zip(actuals, preds)]))


def top3(scores):
    """(n,5) score matrix -> list of top-3 letter lists."""
    return [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in np.asarray(scores)]


def wandb_run(name, metrics, config=None):
    """One W&B run per model. Never allowed to break the pipeline."""
    if not WANDB_ON:
        return
    try:
        wandb.init(project=WANDB_PROJECT, name=name, reinit=True,
                   config=config or {})
        wandb.log({k.lower().replace("@", "").replace("+", "_"): float(v)
                   for k, v in metrics.items()})
        wandb.finish()
    except Exception as e:
        print(f"    (W&B log skipped: {type(e).__name__})")


def evaluate(name, actuals, scores, store=None, log=True, config=None):
    preds = top3(scores)
    t1 = [p[0] for p in preds]
    m = {
        "MAP@3":    map_at_3(actuals, preds),
        "Accuracy": accuracy_score(actuals, t1),
        "MacroF1":  f1_score(actuals, t1, labels=OPTIONS, average="macro", zero_division=0),
        "WgtF1":    f1_score(actuals, t1, labels=OPTIONS, average="weighted", zero_division=0),
    }
    print(f"{name:<28} MAP@3 {m['MAP@3']:.4f} | Acc {m['Accuracy']:.4f} | "
          f"MacroF1 {m['MacroF1']:.4f} | WgtF1 {m['WgtF1']:.4f}")
    if store is not None:
        store[name] = m
    if log:
        wandb_run(name.strip().replace(" ", "-").lower(), m, config)
    return m


RESULTS = {}
print(f"reference: random 0.3667 | prior {PRIOR_MAP3:.4f}")


In [ ]:
# Row positions of the validation split within the cleaned training frame.
# (Hoisted from the source notebook's Model 2 cell -- every model below needs it.)
val_pos = train.index[train["prompt"].isin(va_p)].to_numpy()


## Model 2 - MPNet sentence transformer  *(pre-trained)*

```
   Question --> MPNet --> question embedding (768)
   Options  --> MPNet --> 5 option embeddings (768)
                   |
            cosine similarity
                   |
        rank high -> low --> top 3
```

Embeddings are L2-normalised, so the cosine reduces to a dot product. Nothing is
trained and no corpus is consulted - the model is asked directly which option is
closest in meaning to the question.


In [ ]:
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Encoder choice matters more than anything else here, and it is free on CPU.
#
# all-mpnet-base-v2 is trained for SYMMETRIC similarity ("are these two
# sentences alike?"). This task is ASYMMETRIC -- "does this option ANSWER this
# question?" -- which is what the multi-qa-* models were trained on (they are
# fine-tuned on 215M question/answer pairs). For question->answer matching that
# is usually a large gain over a general-purpose encoder.
#
# Each is evaluated on its own, then all are combined. Trim this list if the
# session is running long; the first entry alone is a reasonable default.
# ---------------------------------------------------------------------------
ENCODERS = [
    "sentence-transformers/multi-qa-mpnet-base-cos-v1",   # QA-tuned  <- expect best
    "sentence-transformers/all-mpnet-base-v2",            # general similarity
    "BAAI/bge-small-en-v1.5",                             # small + strong, fast
    "intfloat/e5-base-v2",                                # different pretraining recipe on CPU
]

ENC = {}          # name -> dict(q_train, o_train, q_test, o_test, scores...)


# bge and e5 are trained WITH instruction prefixes and lose accuracy without
# them: e5 expects "query:" / "passage:", bge expects a retrieval instruction on
# the query side only. Applying the right prefix per family costs nothing.
PREFIX = {
    "bge": ("Represent this sentence for searching relevant passages: ", ""),
    "e5":  ("query: ", "passage: "),
}


def prefixes_for(name):
    low = name.lower()
    for key, pair in PREFIX.items():
        if key in low:
            return pair
    return ("", "")


def embed_with(model, texts, bs=64, prefix=""):
    txt = [prefix + str(t) for t in texts] if prefix else [str(t) for t in texts]
    return normalize(model.encode(txt, batch_size=bs,
                                  show_progress_bar=False, convert_to_numpy=True))


def run_encoder(name):
    print(f"\n--- {name} ---")
    m = SentenceTransformer(name, device=str(DEVICE))
    d = m.get_sentence_embedding_dimension()

    qp, op = prefixes_for(name)
    if qp or op:
        print(f"  using instruction prefixes: query={qp!r} passage={op!r}")

    def enc(df):
        q = embed_with(m, df["prompt"].astype(str), prefix=qp)
        flat = embed_with(m, [str(r[o]) for _, r in df.iterrows() for o in OPTIONS],
                          prefix=op)
        return q.astype(np.float32), flat.reshape(len(df), len(OPTIONS), -1).astype(np.float32)

    qtr, otr = enc(train)
    qte, ote = enc(test)
    del m; gc.collect()

    # cosine(question, option) -- vectors are L2-normalised so this is a dot product
    s_tr = np.einsum("nd,nkd->nk", qtr, otr).astype(np.float32)
    s_te = np.einsum("nd,nkd->nk", qte, ote).astype(np.float32)
    return dict(dim=d, q_train=qtr, o_train=otr, q_test=qte, o_test=ote,
                s_train=s_tr, s_test=s_te)


val_pos = train.index[train["prompt"].isin(va_p)].to_numpy()

for name in ENCODERS:
    try:
        ENC[name] = run_encoder(name)
        short = name.split("/")[-1]
        evaluate(f"2. {short}", y_val, ENC[name]["s_train"][val_pos], RESULTS)
    except Exception as e:
        print(f"  skipped ({type(e).__name__}: {str(e)[:90]})")

assert ENC, "no encoder could be loaded"

# ---- combine encoders: average of within-question z-scored cosines ----
def zrow(a):
    a = np.asarray(a, dtype=np.float64)
    sd = a.std(axis=1, keepdims=True)
    return (a - a.mean(axis=1, keepdims=True)) / np.where(sd > 1e-9, sd, 1.0)


# Weighted blend rather than a plain average: encoders differ in quality, and a
# weak one dragging down a strong one is a real cost. Weights are searched over
# ALL 2000 rows (these signals are unsupervised, so every row is a fair test).
_names = list(ENC)
_z = {n: zrow(ENC[n]["s_train"]) for n in _names}
_rng0 = np.random.default_rng(SEED)
_cands = [{n: 1.0 for n in _names}]
_cands += [{m: (1.0 if m == n else 0.0) for m in _names} for n in _names]
_cands += [{n: float(_rng0.choice([0, 0.5, 1, 2, 3])) for n in _names} for _ in range(300)]

_bw, _bs = None, -np.inf
for w in _cands:
    if sum(w.values()) == 0:
        continue
    sc = map_at_3(y_train, top3(sum(w[n] * _z[n] for n in _names)))
    if sc > _bs:
        _bs, _bw = sc, w
print(f"\nencoder blend weights: { {k: v for k, v in _bw.items() if v} }  -> {_bs:.4f}")

mpnet_train = sum(_bw[n] * _z[n] for n in _names).astype(np.float32)
mpnet_test  = sum(_bw[n] * zrow(ENC[n]["s_test"]) for n in _names).astype(np.float32)
mpnet_val   = mpnet_train[val_pos]

evaluate(f"2. ENCODER BLEND ({len(ENC)})", y_val, mpnet_val, RESULTS)
evaluate("   blend on all 2000 rows", y_train, mpnet_train)
print("   ^ agree because no training corpus is used")

# downstream heads use the strongest single encoder's embeddings
best_enc = max(ENC, key=lambda n: map_at_3(y_val, top3(ENC[n]["s_train"][val_pos])))
print(f"\nStrongest encoder: {best_enc.split('/')[-1]}")
Q_train, O_train = ENC[best_enc]["q_train"], ENC[best_enc]["o_train"]
Q_test,  O_test  = ENC[best_enc]["q_test"],  ENC[best_enc]["o_test"]
EMB_DIM = ENC[best_enc]["dim"]
Q_val, O_val = Q_train[val_pos], O_train[val_pos]


## Model 3 - Deepnet over MPNet pair features  *(supervised)*

```
  [ q ; o ; q*o ; |q-o| ]  (4 x 768)
            |
   Linear(3072 -> 256) + ReLU + Dropout
            |
   Linear(256 -> 64)   + ReLU + Dropout
            |
   Linear(64 -> 1)  ->  score per option
            |
   softmax over 5 options -> cross-entropy
```

The elementwise product and absolute difference are what let the head model
*matching* between question and option rather than the two vectors separately.
Trained 5-fold with `GroupKFold` on cleaned prompts, so every training row gets
an honest out-of-fold prediction.


In [ ]:
class DeepNet(nn.Module):
    def __init__(self, d, hidden=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4 * d, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(p / 2),
            nn.Linear(64, 1))

    def forward(self, q, o):                       # q,o: (B,5,D)
        return self.net(torch.cat([q, o, q * o, (q - o).abs()], -1)).squeeze(-1)


def pair_inputs(Q, O):
    q = torch.tensor(np.repeat(Q[:, None, :], len(OPTIONS), axis=1))
    return q, torch.tensor(O)


def fit_deepnet(Q, O, y, epochs=30, lr=1e-3, bs=32, seed=SEED, d=None,
                hidden=256, drop=0.3):
    torch.manual_seed(seed)
    q, o = pair_inputs(Q, O)
    y = torch.tensor(y, dtype=torch.long)
    model = DeepNet(d or Q.shape[1], hidden=hidden, p=drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    q, o, y = q.to(DEVICE), o.to(DEVICE), y.to(DEVICE)

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(y))
        for k in range(0, len(y), bs):
            b = perm[k:k + bs]
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(q[b], o[b]), y[b])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def deepnet_predict(model, Q, O):
    model.eval()
    q, o = pair_inputs(Q, O)
    return model(q.to(DEVICE), o.to(DEVICE)).cpu().numpy().astype(np.float32)


def run_folds(fit_fn, pred_fn, n_rows, n_test, tag, n_seeds=1):
    """5-fold out-of-fold predictions, averaged over `n_seeds` restarts.

    A single restart of a small network is noisy: the same architecture on the
    same fold can differ by several MAP@3 points through initialisation and
    batch order alone. Averaging a few restarts removes most of that variance
    and is usually worth more than any single hyperparameter change.
    """
    oof = np.zeros((n_rows, len(OPTIONS)), dtype=np.float32)
    tst = np.zeros((n_test, len(OPTIONS)), dtype=np.float32)
    gkf = GroupKFold(n_splits=N_FOLDS)
    groups = train["prompt"].values
    y_idx = np.array([OPTIONS.index(a) for a in train["answer"]])

    for f, (tr_i, va_i) in enumerate(gkf.split(train, groups=groups), 1):
        fold_oof = np.zeros((len(va_i), len(OPTIONS)), dtype=np.float32)
        for sd in range(n_seeds):
            model = fit_fn(tr_i, y_idx[tr_i], SEED + 100 * sd + f)
            fold_oof += pred_fn(model, va_i, "train") / n_seeds
            tst += pred_fn(model, None, "test") / (N_FOLDS * n_seeds)
            del model; gc.collect()
        oof[va_i] = fold_oof
        s = map_at_3([train["answer"].iloc[i] for i in va_i], top3(fold_oof))
        print(f"  {tag} fold {f}/{N_FOLDS}  holdout MAP@3 {s:.4f}"
              + (f"  ({n_seeds} seeds)" if n_seeds > 1 else ""))
    return oof, tst


# --- small hyperparameter search on the validation split (cheap on CPU) ---
print("Tuning deepnet on validation...")
tr_mask = ~np.isin(np.arange(len(train)), val_pos)
tr_idx = np.where(tr_mask)[0]
y_idx_all = np.array([OPTIONS.index(a) for a in train["answer"]])

best_cfg, best_v = None, -np.inf
for hidden in (128, 256):
    for drop in (0.2, 0.4):
        for lr in (1e-3, 3e-3):
            m_ = fit_deepnet(Q_train[tr_idx], O_train[tr_idx], y_idx_all[tr_idx],
                             epochs=20, lr=lr, seed=SEED, d=EMB_DIM,
                             hidden=hidden, drop=drop)
            v = map_at_3(y_val, top3(deepnet_predict(m_, Q_val, O_val)))
            print(f"  hidden={hidden:<4} drop={drop:<4} lr={lr:<6} -> {v:.4f}")
            if v > best_v:
                best_v, best_cfg = v, dict(hidden=hidden, drop=drop, lr=lr)
            del m_; gc.collect()
print(f"best deepnet config: {best_cfg}  (val {best_v:.4f})")

print("Training Deepnet (best config), 5-fold...")
deep_oof, deep_test = run_folds(
    fit_fn=lambda idx, yy, sd: fit_deepnet(Q_train[idx], O_train[idx], yy, seed=sd,
                                       lr=best_cfg["lr"], d=EMB_DIM,
                                       hidden=best_cfg["hidden"],
                                       drop=best_cfg["drop"]),
    pred_fn=lambda m, idx, which: deepnet_predict(
        m, Q_train[idx] if which == "train" else Q_test,
        O_train[idx] if which == "train" else O_test),
    n_rows=len(train), n_test=len(test), tag="deepnet", n_seeds=3)

deep_val = deep_oof[val_pos]
evaluate("3. Deepnet (MPNet)", y_val, deep_val, RESULTS)
evaluate("   Deepnet OOF (2000)", y_train, deep_oof)
